In [1]:
!pip install dash dash-bootstrap-components pandas plotly

In [2]:
import os
import webbrowser
import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.express as px

# ==========================================
# 1. DATA LOADING AND CLEANING
# ==========================================
def load_and_prep_data():
    # Looks directly inside your current local folder for the CSV files
    iapt_file = "psych-ther-ann-rep-csv-2024-25-main2.csv"
    antidepressant_file = "items for antidepressant drugs per.csv"
    anxiolytics_file = "items for anxiolytics per.csv"
    
    # --- Process Prescribing Data ---
    df_anti = pd.read_csv(antidepressant_file)
    df_anx = pd.read_csv(anxiolytics_file)
    
    df_anti['date'] = pd.to_datetime(df_anti['date'])
    df_anx['date'] = pd.to_datetime(df_anx['date'])
    df_anti['Medication_Type'] = 'Antidepressant'
    df_anx['Medication_Type'] = 'Anxiolytic'
    
    # Combine prescribing data
    df_prescribe = pd.concat([df_anti, df_anx], ignore_index=True)
    df_prescribe['name'] = df_prescribe['name'].str.replace('"', '').str.strip()
    
    # --- Process IAPT Data ---
    df_iapt = pd.read_csv(iapt_file)
    df_iapt['Count_ReferralsReceived'] = pd.to_numeric(df_iapt['Count_ReferralsReceived'], errors='coerce')
    df_iapt['Count_AccessingServices'] = pd.to_numeric(df_iapt['Count_AccessingServices'], errors='coerce')
    df_iapt['OrgName'] = df_iapt['OrgName'].str.strip() if 'OrgName' in df_iapt.columns else ""
    
    return df_prescribe, df_iapt

df_prescribe, df_iapt = load_and_prep_data()
available_regions = sorted(df_prescribe['name'].unique())
min_date = df_prescribe['date'].min()
max_date = df_prescribe['date'].max()

# ==========================================
# 2. DASH APPLICATION LAYOUT
# ==========================================
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.Div([
            html.H1("Mental Health Services & Prescribing Analytics", className="text-white p-3 mb-2 bg-primary rounded-top text-center"),
            html.P("Interactive analysis of NHS IAPT metrics and community prescribing trends.", className="text-muted text-center lead")
        ]), width=12)
    ], className="mt-4"),
    
    dbc.Row([
        dbc.Col([
            html.Label("Select NHS Region/Organization Name:", className="fw-bold"),
            dcc.Dropdown(
                id='region-dropdown',
                options=[{'label': r, 'value': r} for r in available_regions],
                value=available_regions[0] if available_regions else None,
                clearable=False,
                className="mb-3"
            )
        ], md=6, lg=6),
        
        dbc.Col([
            html.Label("Select Date Range:", className="fw-bold"),
            dcc.DatePickerRange(
                id='date-picker',
                min_date_allowed=min_date, max_date_allowed=max_date,
                start_date=min_date, end_date=max_date,
                display_format='YYYY-MM-DD', style={"width": "100%"}
            )
        ], md=6, lg=6)
    ], className="bg-light p-3 rounded mb-4 shadow-sm"),
    
    dbc.Row([
        dbc.Col(dbc.Card([dbc.CardBody([html.H5("Total Prescribed Items", className="card-title text-secondary"), html.H3(id="kpi-total-items", className="card-text text-primary")])], color="light"), sm=4),
        dbc.Col(dbc.Card([dbc.CardBody([html.H5("Actual Direct Cost", className="card-title text-secondary"), html.H3(id="kpi-total-cost", className="card-text text-success")])], color="light"), sm=4),
        dbc.Col(dbc.Card([dbc.CardBody([html.H5("Matched IAPT Referrals (Annual Est.)", className="card-title text-secondary"), html.H3(id="kpi-iapt-referrals", className="card-text text-info")])], color="light"), sm=4),
    ], className="mb-4"),
    
    dbc.Row([
        dbc.Col([html.H4("Prescription Volume Trends Over Time", className="p-2 mb-0 bg-secondary text-white rounded-top"), dcc.Graph(id='trend-time-series', className="border rounded-bottom bg-white shadow-sm")], width=12, lg=7, className="mb-4"),
        dbc.Col([html.H4("Cost & Drug Type Split", className="p-2 mb-0 bg-secondary text-white rounded-top"), dcc.Graph(id='drug-type-pie', className="border rounded-bottom bg-white shadow-sm")], width=12, lg=5, className="mb-4")
    ]),
    
    dbc.Row([
        dbc.Col([html.H4("Regional Efficiency: Items vs. Actual Cost", className="p-2 mb-0 bg-secondary text-white rounded-top"), dcc.Graph(id='cost-volume-scatter', className="border rounded-bottom bg-white shadow-sm")], width=12)
    ], className="mb-5")
], fluid=True)

# ==========================================
# 3. INTERACTIVE CALLBACK LOGIC
# ==========================================
@app.callback(
    [Output('trend-time-series', 'figure'), Output('drug-type-pie', 'figure'), Output('cost-volume-scatter', 'figure'),
     Output('kpi-total-items', 'children'), Output('kpi-total-cost', 'children'), Output('kpi-iapt-referrals', 'children')],
    [Input('region-dropdown', 'value'), Input('date-picker', 'start_date'), Input('date-picker', 'end_date')]
)
def update_dashboard(selected_region, start_date, end_date):
    filtered_prescribe = df_prescribe[
        (df_prescribe['name'] == selected_region) & 
        (df_prescribe['date'] >= pd.to_datetime(start_date)) & 
        (df_prescribe['date'] <= pd.to_datetime(end_date))
    ]
    
    if 'OrgName' in df_iapt.columns:
        matched_iapt = df_iapt[df_iapt['OrgName'].str.contains(selected_region, case=False, na=False)]
        iapt_referrals = matched_iapt['Count_ReferralsReceived'].dropna().sum()
        iapt_text = f"{int(iapt_referrals):,}" if iapt_referrals > 0 else "N/A"
    else:
        iapt_text = "N/A"
    
    total_items = filtered_prescribe['y_items'].sum()
    total_cost = filtered_prescribe['y_actual_cost'].sum()

    time_agg = filtered_prescribe.groupby(['date', 'Medication_Type'])['y_items'].sum().reset_index()
    fig_time = px.line(time_agg, x='date', y='y_items', color='Medication_Type', markers=True, template="plotly_white")
    fig_time.update_layout(margin=dict(l=40, r=40, t=30, b=40))

    type_agg = filtered_prescribe.groupby('Medication_Type')['y_actual_cost'].sum().reset_index()
    fig_pie = px.pie(type_agg, values='y_actual_cost', names='Medication_Type', hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
    fig_pie.update_layout(margin=dict(l=20, r=20, t=30, b=20))
    
    scatter_data = df_prescribe[
        (df_prescribe['date'] >= pd.to_datetime(start_date)) & (df_prescribe['date'] <= pd.to_datetime(end_date))
    ].groupby('name').agg({'y_items': 'sum', 'y_actual_cost': 'sum'}).reset_index()
    scatter_data['Highlight'] = scatter_data['name'].apply(lambda x: 'Selected Region' if x == selected_region else 'Other Regions')
    
    fig_scatter = px.scatter(scatter_data, x='y_items', y='y_actual_cost', color='Highlight', hover_name='name',
                             color_discrete_map={'Selected Region': '#E74C3C', 'Other Regions': '#BDC3C7'}, template="plotly_white")
    fig_scatter.update_traces(marker=dict(size=12, opacity=0.8))

    return fig_time, fig_pie, fig_scatter, f"{int(total_items):,}", f"£{total_cost:,.2f}", iapt_text

# ==========================================
# 4. RUN SERVER FOR LOCAL JUPYTER
# ==========================================
if __name__ == '__main__':
    # Define local network tracking coordinates
    port = 8050
    url = f"http://127.0.0.1:{port}/"
    
    # Automatically triggers your standard web browser to launch a clean tab
    webbrowser.open_new(url)
    print(f"\nDashboard launched! If it didn't open automatically, navigate to: {url}")
    
    # Start server framework (use_reloader=False prevents notebook core crashes)
    app.run(port=port, debug=False, use_reloader=False)

C:\Users\Admin\AppData\Local\Temp\ipykernel_14832\2868247414.py:32: DtypeWarning: Columns (15,31,32,33,40,41,42,43,44,45,46,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df_iapt = pd.read_csv(iapt_file)



Dashboard launched! If it didn't open automatically, navigate to: http://127.0.0.1:8050/


[2026-05-24 21:34:26,895] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "C:\Users\Admin\anaconda3\Lib\site-packages\flask\app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
  File "C:\Users\Admin\anaconda3\Lib\site-packages\flask\app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "C:\Users\Admin\anaconda3\Lib\site-packages\flask\app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\Users\Admin\anaconda3\Lib\site-packages\flask\app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "C:\Users\Admin\anaconda3\Lib\site-packages\dash\_get_app.py", line 17, in wrap
    return ctx.run(func, self, *args, **kwargs)
           ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C: